# Accessing Treasury Bill Prices From Treasury Direct Website

## Accessing price data directly from the FedInvest



Price data for U.S. Treasury securities is available on the FedInvest page of the Treasury Direct website. This data includes the current day's prices (updated after 1 pm New York time) and daily historical prices dating back to September 2008.

The current page design is straightforward, making the data easily accessible using Pandas. As shown below, the page features a form to select the desired date for the prices.

A specialized function, <font color='green'>FEDInvest</font>, is used to retrieve this data. The function takes the requested date as its single argument and returns the data as a Pandas DataFrame. Clicking the accompanying image provides a brief explanation of the website and the function.

[![FEDInvest Video](https://img.youtube.com/vi/YuhiFDyvNb0/maxresdefault.jpg)](https://www.youtube.com/watch?v=YuhiFDyvNb0)




## Importing libraries, modules, and functions


This notebook illustrates how to obtain daily price data for U.S. Treasury Securities. It utilizes a function imported from a custom module. The necessary modules are minimal and primarily consist of those found in the standard Python library, along with IPython and Pandas. IPython is included by default in both Jupyter and Colab environments, and no installation is required.


In [ ]:
import sys
import requests
from datetime import date,datetime
from types import ModuleType
try:
    import pandas as pd
except:
    !pip install pandas
    import pandas as pd

## Getting FEDInvest from the custom module

In [ ]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
  response=requests.get(url)
  # Raise an exception for bad status codes (like 404 Not Found)
  response.raise_for_status()
  module= ModuleType(module_name)
  #Code contained in response.text executed
  exec(response.text, module.__dict__)
  # Module added to sys
  sys.modules[module_name]=module
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")

# Now that 'basic_concepts_fixed_income' exists in the notebook, import the specific functions
from basic_concepts_fixed_income import (FEDInvest)

## <font color='green'>FEDInvest</font>

The function accepts a single argument: the desired date for the prices. Unlike previous chapters, this function uses the <font color='green'>post</font>
 method of the request module instead of the <font color='green'>get</font>
 method.

The initial webpage contains a form for the day, month, and year of the price data. This information is sent to the page using the <font color='green'>post</font>
 method. We define a dictionary containing the day, month, and year included in the <font color='green'>post</font>
 request. The required data types and names are determined by inspecting the page (using F12).

The response to this input is a page that contains a table of price data. We use the Pandas <font color='green'>read_html</font>
 function that searches the response text for tables. By using the <font color='green'>match</font>
 parameter, we restrict the returned results to only those tables that contain the value assigned to <font color='green'>match</font>. In this case, only tables that include 'CUSIP' are returned, resulting in a single table.


```
 # URL address of Treasury Direct Select A Date
  url = "https://treasurydirect.gov/GA-FI/FedInvest/selectSecurityPriceDate"

 #  variable names and type identified from inspecting url
  month=str(price_date.month)
  day=str(price_date.day)
  year=str(price_date.year)

  # payload passed in request post
  payload={'priceDate.month':month,
           'priceDate.day':day,
           'priceDate.year':year,
           "submit": "Show Prices"}

  # fires off form and returns prices for date
  try:
        response = requests.post(url, data=payload, headers=headers, timeout=10)
        response.raise_for_status()
  except requests.exceptions.RequestException as e:
        return f"Connection Error: {e}", None

  # reads the html
  # Pandas recommends to wrap the response in StingIO to make text file like
  tables=pd.read_html(StringIO(response.text),match='CUSIP')


  # from inspection there is a single table
  return tables[0], price_date,settlement_date        
```

[The function is available here](https://patrickjhess.github.io/Imported-Functions/FEDInvest.html#fedinvest-is-a-helper-function-that-gets-daily-price-data-for-u-s-treasury-securities)


## Getting data from FEDInvest for January 17$^{th}$ 2025

Chapter Four downloaded bond data from dropbox for settlement on January 21$^{st}$ 2025. The maturity dates of that data is between January 28, 2025 and January 31, 2030. That dataset had 309 bills, notes, or bonds.

In [ ]:
# set the date for prices
price_date=date(2025,1,17)

# fetch the data
df,date_stamp,settlement=FEDInvest(price_date)

#display results
display(df)
display(f' Requested Date; {date_stamp}; Settlement Date {settlement}' )

,CUSIP,SECURITY TYPE,RATE,MATURITY DATE,CALL DATE,BUY,SELL,END OF DAY
0,912797MY0,MARKET BASED BILL,0.000%,01/21/2025,NaN,0.000000,99.953111,100.000000
1,912797JR9,MARKET BASED BILL,0.000%,01/23/2025,NaN,0.000000,99.929500,99.976556
2,912797MZ7,MARKET BASED BILL,0.000%,01/28/2025,NaN,99.871208,99.871056,99.918139
3,912797LZ8,MARKET BASED BILL,0.000%,01/30/2025,NaN,99.847792,99.847611,99.894500
4,912797NF0,MARKET BASED BILL,0.000%,02/04/2025,NaN,99.788750,99.788500,99.835500
...,...,...,...,...,...,...,...,...
448,91282CJD4,MARKET BASED FRN,4.502%,10/31/2025,NaN,100.089107,100.066017,100.071788
449,91282CJU6,MARKET BASED FRN,4.578%,01/31/2026,NaN,100.163356,100.143067,100.150673
450,91282CKM2,MARKET BASED FRN,4.482%,04/30/2026,NaN,100.073564,100.048611,100.054847
451,91282CLA7,MARKET BASED FRN,4.514%,07/31/2026,NaN,100.135440,100.113131,100.113131


' Requested Date; 2025-01-17; Settlement Date 2025-01-21'

Chapter Four downloaded bond data from dropbox for settlement on January 21, 2025. The maturity dates of that data is between January 28, 2025 and January 31, 2030. That dataset had 309 bills, notes, or bonds.  This dataset has 452 securities, but some of those are floating rate notes (FRN) or inflation indexed securities (TIPS).

## Cleaning up the data

### Determining types of securities included in the DataFrame
We will focus on analyzing bills, notes, and bonds. The initial step is to identify the unique types of instruments within the DataFrame. We will accomplish this by using the Pandas <font color='green'>duplicated</font> method, which indicates whether a row is a duplicate. By setting the subset parameter to 'SECURITY TYPE', we target that specific column. We then negate the returned boolean values with the ~ symbol to isolate and return only the first occurrence of each unique security type.


In [ ]:
# find rows that are unique (return False for duplicated)
unique_type=~df.duplicated(subset=['SECURITY TYPE'])
display(df[unique_type])

,CUSIP,SECURITY TYPE,RATE,MATURITY DATE,CALL DATE,BUY,SELL,END OF DAY
0,912797MY0,MARKET BASED BILL,0.000%,01/21/2025,NaN,0.0,99.953111,100.000000
49,9128283V0,MARKET BASED NOTE,2.500%,01/31/2025,NaN,0.0,99.937500,99.937500
292,912810ET1,MARKET BASED BOND,7.625%,02/15/2025,NaN,0.0,99.937500,99.968750
394,912828ZJ2,TIPS,0.125%,04/15/2025,NaN,0.0,99.781250,99.781250
445,91282CGF2,MARKET BASED FRN,4.533%,01/31/2025,NaN,0.0,100.002911,100.003051


## Retain bills, notes, and bonds, set index,convert 'RATE' to coupon, drop and rename columns

* Retain bills, notes, and bonds
  *   <font color='green'>rows_to_keep</font>: This refers to the specific types of securities that will be retained.
  *   <font color='green'>isin</font>: This parameter return True if the value is present in <font color='green'>rows_to_keep</font>
  *   <font color='green'>copy</font>:  This allows reassigning the modified DataFrame back to its original name.

* Make index Pandas timestamp of 'MATURITY DATE' and sort.

  *   <font color='green'>set_index</font>: makes the 'MATURITY DATE' the index of the DataFrame

  *   <font color='green'>pd.to_dateteime</font>: convert the index to Pandas datetime.

  *   <font color='green'>sort_index</font>: sorts the DataFrame by ascending maturity date.

* Convert rate to coupon payment for one hundred par value

  *   <font color='green'>str.replace</font>: replaces every element in the series.
  *   <font color='green'>as.type</font>: converts every element in the string into a floating point number

* Drop columns and change column names

  *  <font color='green'>drop</font>: method drops 'CUSIP' and 'CALL DATE' columns.
  * <font color='green'>rename</font>: method changes column names.
  * <font color='green'>change_column_names</font>: a dictionary that maps the old names as keys to the new names the values.


In [ ]:
# define the securities we want to keep
rows_to_keep=['MARKET BASED BILL','MARKET BASED NOTE','MARKET BASED BOND']

# use isin to generate a seroes of True/False keep True and reassigne the DataFrame
df=df[df['SECURITY TYPE'].isin(rows_to_keep)].copy()

# make 'MATURITY DATE' column index, convert to timestamp, and sort chronological
df.set_index('MATURITY DATE',inplace=True)
df.index=pd.to_datetime(df.index)
df.sort_index(inplace=True)

# replace '%' with nothing and cast results as floating point
df['RATE']=df['RATE'].str.replace('%','').astype(float)

# define the columnes to be dropped
drop_columns=['CUSIP','CALL DATE']
df.drop(columns=drop_columns,inplace=True)

# define the dictionary that maps new names (values) to old (keys)
change_column_names={'RATE':'Coupon',
                     'BUY':'Price Ask',
                     'SELL':'Price Bid'}
df.rename(columns=change_column_names,inplace=True)

display(df)

,SECURITY TYPE,Coupon,Price Ask,Price Bid,END OF DAY
MATURITY DATE,,,,,
2025-01-21,MARKET BASED BILL,0.000,0.000000,99.953111,100.000000
2025-01-23,MARKET BASED BILL,0.000,0.000000,99.929500,99.976556
2025-01-28,MARKET BASED BILL,0.000,99.871208,99.871056,99.918139
2025-01-30,MARKET BASED BILL,0.000,99.847792,99.847611,99.894500
2025-01-31,MARKET BASED NOTE,4.125,0.000000,100.000000,100.000000
...,...,...,...,...,...
2053-11-15,MARKET BASED BOND,4.750,98.312500,98.281250,98.250000
2054-02-15,MARKET BASED BOND,4.250,90.578125,90.562500,90.531250
2054-05-15,MARKET BASED BOND,4.625,96.468750,96.437500,96.437500


## <font color='green'>Application: Access FEDInvest data for June 1$^{st}$ 2010 and create a DataFrame with an index equal to maturity dates as Pandas timestamps.</font>

 see [Chapter Nine Hints: Access FEDInvest data for June 1 2010 and create a DataFrame](https://patrickjhess.github.io/Hints-Results/Chapter_Nine_Hints.html#chapter-nine-hints), and check the [expected results here](https://patrickjhess.github.io/Hints-Results/Chapter_Nine_Results.html#chapter-nine-results).

